# S06-13A (Student) — Building Graph from Rhino OBJ

Model the building in **Rhino**. Export **four OBJ files**:

| File | Layer |
|------|--------|
| `ground.obj` | slab / podium |
| `columns.obj` | columns |
| `offices.obj` | office volumes |
| `core.obj` | core + corridors |

Store them in `Supporting Files/` (or change `SUPPORT_DIR` in the setup cell).

---

## Pipeline

1. **Import** OBJs → `Topology.ByOBJPath`
2. **Tag** each cell (`cell_type` selectors)
3. **CellComplex** → merge all cells (`Topology.SelfMerge`)
4. **Transfer** dictionaries onto cells
5. **Graph** → `Graph.ByTopology` + one-hot features
6. **Export** CSV → `Graph.ExportToCSV`
7. **Predict** → **S06-13** (Phase 2), `pyg_model.pt`


Reference outputs (gray) below = example building; yours will differ.


### Setup

Install `topologicpy` if needed. Import `Topology`, `Cluster`, `Dictionary`, `Graph`, `Helper`.

In [28]:
# !pip install topologicpy

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Color import Color
from topologicpy.Helper import Helper
from topologicpy.Plotly import Plotly
from pathlib import Path

obj_candidates = [Path("..") / "obj", Path("assignment03") / "obj", Path("obj")]
SUPPORT_DIR = next((path.resolve() for path in obj_candidates if path.is_dir()), None)
if SUPPORT_DIR is None:
    raise FileNotFoundError("Could not find the assignment03/obj folder")


### 1. Import parts

`Topology.ByOBJPath` for each OBJ. **TODO:** your four paths.

In [29]:
# TODO: Import geometry, use code below as an example

ground_objs = Topology.ByOBJPath(str(SUPPORT_DIR / "ground.obj"), transposeAxes=False)
column_objs = Topology.ByOBJPath(str(SUPPORT_DIR / "columns.obj"), transposeAxes=False)
office_objs = Topology.ByOBJPath(str(SUPPORT_DIR / "offices.obj"), transposeAxes=False)
core_objs = Topology.ByOBJPath(str(SUPPORT_DIR / "core.obj"), transposeAxes=False)


**Check:** `Topology.Show(ground_objs, office_objs, core_objs, column_objs)`

In [30]:
# TODO: Topology.Show(ground_objs, office_objs, core_objs, column_objs)

### 2. Assign categories

Per layer: Faces → Flatten → SelfMerge → `Topology.Cells`.  
For each cell: `Dictionary` with `cell_type`, `cell_name`, `cell_color`; `InternalVertex` + `SetDictionary` → `selectors`.

In [31]:
def cells_from_objects(objects, cell_type, cell_name, cell_color):
    faces = [
        Topology.Faces(obj)
        for obj in objects
        if Topology.IsInstance(obj, "Topology")
    ]
    faces = Helper.Flatten(faces)
    if not faces:
        raise ValueError(f"No faces were imported for {cell_name}")

    merged = Topology.SelfMerge(Cluster.ByTopologies(faces))
    cells = Topology.Cells(merged)
    if not cells:
        raise ValueError(f"No cells could be created for {cell_name}")

    tagged_selectors = []
    for cell in cells:
        dictionary = Dictionary.ByKeysValues(
            ["cell_type", "cell_name", "cell_color"],
            [cell_type, cell_name, cell_color],
        )
        selector = Topology.InternalVertex(cell)
        selector = Topology.SetDictionary(selector, dictionary)
        tagged_selectors.append(selector)
    return cells, tagged_selectors


ground_cells, ground_selectors = cells_from_objects(
    ground_objs, 0, "ground", "green"
)
column_cells, column_selectors = cells_from_objects(
    column_objs, 1, "column", "gray"
)
office_cells, office_selectors = cells_from_objects(
    office_objs, 3, "office", "blue"
)
core_cells, core_selectors = cells_from_objects(
    core_objs, 4, "core", "red"
)

selectors = (
    ground_selectors
    + column_selectors
    + office_selectors
    + core_selectors
)

print(
    "Cells:",
    {
        "ground": len(ground_cells),
        "column": len(column_cells),
        "office": len(office_cells),
        "core": len(core_cells),
    },
)


Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: cells_from_objects
Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: cells_from_objects
Cells: {'ground': 1, 'column': 36, 'office': 17, 'core': 1}


### 3. CellComplex

`all_cells` → `model = Topology.SelfMerge(Cluster.ByTopologies(all_cells))`

In [32]:
all_cells = (
    ground_cells
    + column_cells
    + office_cells
    + core_cells
)
if not all_cells:
    raise ValueError("No cells are available to build the model")

model = Topology.SelfMerge(Cluster.ByTopologies(all_cells))
if model is None:
    raise ValueError("The cells could not be merged into a model")

print(model)
print("Merged cells:", len(Topology.Cells(model)))


Merged cells: 55


### 4. Transfer dictionaries

`Topology.TransferDictionariesBySelectors(model, selectors, tranCells=True)`

In [33]:
model = Topology.TransferDictionariesBySelectors(
    model,
    selectors,
    tranCells=True,
)
if model is None:
    raise ValueError("Dictionary transfer failed")

print("Transferred dictionaries to", len(Topology.Cells(model)), "cells")


Transferred dictionaries to 55 cells


### 4A. Export topology image

Display the coloured CellComplex and save `assignment03/topology.png`.


In [ ]:
TOPOLOGY_PNG = SUPPORT_DIR.parent / "topology.png"

topology_figure = Topology.Show(
    Topology.Cells(model),
    showVertices=False,
    showEdges=True,
    edgeColor="black",
    edgeWidth=1,
    showFaces=True,
    faceColorKey="cell_color",
    faceOpacity=1,
    width=1220,
    height=620,
    backgroundColor="white",
    camera=[1.5, -1.5, 1.2],
    center=[0, 0, 0],
    up=[0, 0, 1],
    projection="perspective",
    showFigure=False,
)

image_status = Plotly.FigureExportToPNG(
    topology_figure,
    str(TOPOLOGY_PNG),
    width=1220,
    height=620,
    overwrite=True,
)

print("Image export status:", image_status)
print("Topology image:", TOPOLOGY_PNG.resolve())
topology_figure.show()


### 5. Adjacency graph

`Graph.ByTopology(model)` + one-hot `feature_00`…`feature_04` from `cell_type`.

In [34]:
def one_hot_encode(value, n):
    value = int(value)
    if value < 0 or value >= n:
        raise ValueError(f"value must be in [0, {n - 1}], got {value}")

    encoded = [0] * n
    encoded[value] = 1
    return encoded


graph = Graph.ByTopology(model)
if graph is None:
    raise ValueError("Graph.ByTopology could not create a graph from model")

vertices = Graph.Vertices(graph)
feature_names = [f"feature_{i:02d}" for i in range(5)]

for vertex in vertices:
    dictionary = Topology.Dictionary(vertex)
    cell_type = Dictionary.ValueAtKey(dictionary, "cell_type")
    if cell_type is None:
        raise ValueError("A graph vertex is missing its cell_type dictionary value")

    encoded = one_hot_encode(cell_type, len(feature_names))
    for feature_name, feature_value in zip(feature_names, encoded):
        dictionary = Dictionary.SetValueAtKey(
            dictionary,
            feature_name,
            feature_value,
        )
    Topology.SetDictionary(vertex, dictionary)

print("Graph vertices:", len(vertices))
print("Graph edges:", len(Graph.Edges(graph)))
print("Node features:", feature_names)

for vertex in vertices:
    dictionary = Topology.Dictionary(vertex)
    print(Dictionary.Keys(dictionary), Dictionary.Values(dictionary))


Graph vertices: 55
Graph edges: 143
Node features: ['feature_00', 'feature_01', 'feature_02', 'feature_03', 'feature_04']
['aabb', 'category', 'cell_color', 'cell_name', 'cell_type', 'feature_00', 'feature_01', 'feature_02', 'feature_03', 'feature_04'] [[24.154663, 4.0, -0.209616, 24.354664, 8.0, -0.009616], 0, 'gray', 'column', 1, 0, 1, 0, 0, 0]
['aabb', 'category', 'cell_color', 'cell_name', 'cell_type', 'feature_00', 'feature_01', 'feature_02', 'feature_03', 'feature_04'] [[10.954663, 4.0, -8.008121, 11.154663, 8.0, -7.808121], 0, 'gray', 'column', 1, 0, 1, 0, 0, 0]
['aabb', 'category', 'cell_color', 'cell_name', 'cell_type', 'feature_00', 'feature_01', 'feature_02', 'feature_03', 'feature_04'] [[16.234663, 4.0, -8.008121, 16.434664, 8.0, -7.808121], 0, 'gray', 'column', 1, 0, 1, 0, 0, 0]
['aabb', 'category', 'cell_color', 'cell_name', 'cell_type', 'feature_00', 'feature_01', 'feature_02', 'feature_03', 'feature_04'] [[18.874664, 4.0, -8.008121, 19.074663, 8.0, -7.808121], 0, 'gray'

### 6. Export CSV

`Graph.ExportToCSV` → folder with `graphs.csv`, `nodes.csv`, `edges.csv`.

In [35]:
EXPORT_DIR = SUPPORT_DIR.parent / "bgr_dataset"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

status = Graph.ExportToCSV(
    graph,
    path=str(EXPORT_DIR),
    nodeFeaturesKeys=feature_names,
    overwrite=True,
)

print("Export status:", status)
print("Export directory:", EXPORT_DIR)
print("Exported files:", [path.name for path in EXPORT_DIR.glob("*.csv")])


Export status: True
Export directory: C:\Users\danie\Documents\GitHub\term 3\graph ml\Graph ML\assignment03\bgr_dataset
Exported files: ['edges.csv', 'graphs.csv', 'nodes.csv']


### 7. Predict (S06-13)

**S06-13 GML Graph Classification** → Phase 2: set `dataset_dir` to your export folder, `LoadModel(pyg_model.pt)`, `Predict()` → label 0–4.